## Faithfulness 评估总流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    CTX[检索上下文 Context] --> JUDGE
    ANS[模型答案 Answer] --> EXT[Claim 提取]
    EXT --> CL[原子声明列表]
    CL --> JUDGE[逐条 LLM-Judge 验证]
    CTX --> JUDGE
    JUDGE --> SC[Faithfulness = 支持数 / 总数]
```

---
# Part 1 · 环境准备

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

api_key = "sk-your-api-key-here"

---
# Chapter 4 · Faithfulness 深入

## Faithfulness 挑战：为什么 LLM 会幻觉

**Faithfulness 定义**：答案中的每一个事实性陈述，都能在检索到的上下文中找到明确支持。

| 来源 | 描述 | 示例 |
|------|------|------|
| **参数记忆污染** | LLM 用训练数据中的知识代替上下文信息 | 知识库说"2024年"，LLM 回答"2023年" |
| **推理跳跃** | LLM 在 chunk 信息之间做了上下文没支持的推断 | "A导致B，B导致C" → LLM说"A直接导致C" |
| **Prompt 格式歧义** | Prompt 没有明确要求"只使用提供的上下文" | LLM 自由发挥，混入训练知识 |

**解决方案**：将 Faithfulness 分解为可验证的**原子性 Claim**，再逐一用 NLI 或 LLM-Judge 验证。

## 两阶段算法

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    A[Answer] --> S1[阶段1: Claim Extraction]
    S1 --> C1[Claim 1]
    S1 --> C2[Claim 2]
    S1 --> CN[Claim N]
    C1 --> S2[阶段2: 逐条验证]
    C2 --> S2
    CN --> S2
    CTX[Context] --> S2
    S2 --> V[supported true/false]
```

## Claim 提取步骤

**核心思想**：将复杂答案拆解成不可再分的最小事实单元。

**示例答案**
> "Transformer 模型由 Vaswani 等人在 2017 年提出，采用自注意力机制，彻底改变了 NLP 领域。与 RNN 相比，Transformer 可以并行计算，训练速度提升了约 3 倍。"

**提取出的原子 Claims**

```
Claim 1: Transformer 模型由 Vaswani 等人提出
Claim 2: Transformer 在 2017 年提出
Claim 3: Transformer 采用自注意力机制
Claim 4: Transformer 改变了 NLP 领域
Claim 5: Transformer 可以并行计算
Claim 6: 与 RNN 相比，Transformer 训练速度提升了约 3 倍
```

**提取原则**：每个 Claim 只包含一个事实；包含数字、人名、时间的 Claim 要单独提取；无法验证的评价性陈述可以跳过。

## Claim 提取流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    ANS[完整 Answer] --> LLM1[LLM Claim Extraction Prompt]
    LLM1 --> JSON[JSON 声明数组]
    JSON --> ATOM[每条 = 一个事实]
```

## NLI 验证方法

**Natural Language Inference（自然语言推理）**

给定前提（Premise）和假设（Hypothesis），判断：

- **Entailment（蕴含）**：前提支持假设 → Claim 被验证
- **Contradiction（矛盾）**：前提反驳假设 → 幻觉
- **Neutral（中立）**：前提不支持也不反驳 → Claim 无法验证

在 Faithfulness 中：`Premise = 检索上下文`，`Hypothesis = 从答案提取的 Claim`。

> NLI 模型优点：速度快，无 API 成本；缺点：中文支持弱。本 Notebook 使用 **LLM-as-Judge** 替代。

## NLI 三分类

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    P[Premise = Context] --> NLI[NLI 模型]
    H[Hypothesis = Claim] --> NLI
    NLI --> E[Entailment 支持]
    NLI --> C[Contradiction 矛盾]
    NLI --> N[Neutral 无法验证]
```

Faithfulness 计分通常只把 **Entailment** 算作 supported。

## LLM-as-Judge Prompt 详解

**判断标准**

- **SUPPORTED**：上下文中有明确的文字或等价表述支持该陈述
- **NOT_SUPPORTED**：上下文中找不到支持该陈述的证据，或该陈述与上下文矛盾
- **PARTIALLY_SUPPORTED**：上下文部分支持，但陈述超出了上下文范围

**关键设计点**

- `PARTIALLY_SUPPORTED` 类别：避免非黑即白，捕捉边界情况
- 要求引用 `evidence`：强迫 LLM-Judge 定位证据，减少误判
- `temperature=0`：确保判断稳定可复现

## LLM-Judge 单条验证

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    CTX[Context] --> LLM[LLM-Judge]
    CL[单条 Claim] --> LLM
    LLM --> SUP[SUPPORTED]
    LLM --> NS[NOT_SUPPORTED]
    LLM --> PS[PARTIALLY_SUPPORTED]
```

本 Notebook 简化为 `supported: true / false` + `reason`。

## Faithfulness 计算

$$\text{Faithfulness} = \frac{\text{verified\_claims}}{\text{total\_claims}}$$

| Claim | LLM-Judge 判断 | 计入验证？ |
|-------|--------------|----------|
| Transformer 在 2017 年提出 | SUPPORTED | 是 |
| Vaswani 等人提出 | SUPPORTED | 是 |
| 采用自注意力机制 | SUPPORTED | 是 |
| 改变了 NLP 领域 | NOT_SUPPORTED | 否 |
| 训练速度提升约 3 倍 | NOT_SUPPORTED | 否 |

$$\text{Faithfulness} = \frac{3}{5} = 0.60$$

> Faithfulness = 0.6 意味着 40% 的陈述没有上下文支撑（建议目标 ≥ 0.85）

## 得分汇总流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    V1[Claim1 supported?] --> CNT[supported_count]
    V2[Claim2 supported?] --> CNT
    VN[ClaimN supported?] --> CNT
    CNT --> DIV[除以 total_claims]
    DIV --> F[Faithfulness 0~1]
```

## Faithfulness 常见陷阱

**陷阱 1：Claims 粒度太粗**

```python
# 错误：一个 Claim 包含多个事实
claim = "Transformer 在 2017 年由 Vaswani 提出并采用了注意力机制"

# 正确：每个 Claim 只有一个事实
claims = [
    "Transformer 在 2017 年提出",
    "Transformer 由 Vaswani 提出",
    "Transformer 采用了注意力机制",
]
```

**陷阱 2：LLM-Judge 的 Prompt 过于宽松**——必须要求引用具体证据，否则 LLM 容易"脑补"支持。

**陷阱 3：对评价性陈述也做 Faithfulness 验证**——"改变了 NLP 领域"是价值判断，会人为拉低分数。解决：在 Claim 提取阶段过滤评价性陈述。

---
# Part 2 · 评估场景与 Prompt 模板（代码实战）

| 场景 | 预期分数 | 说明 |
|------|----------|------|
| A · 忠实答案 | 高 | 答案完全来自上下文 |
| B · 幻觉答案 | 低 | 包含上下文中不存在的事实 |
| C · 部分忠实 | 中 | 部分声明有依据，部分无依据 |

In [2]:
SCENARIOS = [
    {
        "name": "场景 A：忠实答案（高分预期）",
        "context": (
            "RAG 技术通过在生成前先检索相关文档，将外部知识注入到大语言模型的上下文中，"
            "从而减少幻觉、提升回答的准确性。向量数据库存储文档的嵌入向量，"
            "用于相似度检索。"
        ),
        "answer": (
            "RAG 技术先检索相关文档，再注入到模型上下文中，"
            "通过引入外部知识来减少大语言模型的幻觉现象，提升准确性。"
        ),
    },
    {
        "name": "场景 B：幻觉答案（低分预期）",
        "context": (
            "RAG 技术通过在生成前先检索相关文档，将外部知识注入到大语言模型的上下文中，"
            "从而减少幻觉、提升回答的准确性。向量数据库存储文档的嵌入向量，"
            "用于相似度检索。"
        ),
        "answer": (
            "RAG 技术于 2015 年由 OpenAI 发明，并在 GPT-4 中首次应用。"
            "它通过量子计算加速检索，每秒可处理 10 亿个文档。"
            "目前已被 99% 的 AI 公司采用。"
        ),
    },
    {
        "name": "场景 C：部分忠实答案（中分预期）",
        "context": (
            "Reranker 模型对初步检索结果重新排序，提升 Context Precision。"
            "常见的 Reranker 包括 Cohere Rerank 和 BGE-Reranker。"
        ),
        "answer": (
            "Reranker 可以提升检索精度，对结果重新排序是其核心功能。"
            "此外，Reranker 还能自动扩充知识库并生成新文档。"
        ),
    },
]

CLAIM_EXTRACTION_PROMPT = """你是一位文本分析专家。请将以下答案拆解为独立的原子性声明（每条声明只包含一个事实）。

答案：
{answer}

请以 JSON 数组格式返回声明列表，每条声明为一个字符串。示例：
["声明1", "声明2", "声明3"]

只返回 JSON 数组，不要包含任何其他文字或 markdown 代码块标记。"""

CLAIM_VERIFICATION_PROMPT = """你是一位严格的事实核查员。请判断以下声明是否可以从给定的上下文中得到支持。

上下文：
{context}

声明：
{claim}

请返回一个 JSON 对象，包含以下字段：
- "supported": true 或 false
- "reason": 一句话说明理由

只返回 JSON 对象，不要包含任何其他文字或 markdown 代码块标记。"""

---
# Part 3 · 核心函数

**Faithfulness = 被支持的声明数 / 总声明数**

## 代码实现流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    ANS[answer] --> EC[extract_claims]
    EC --> LOOP[for each claim]
    LOOP --> VC[verify_claim]
    CTX[context] --> VC
    VC --> SUM[supported_count++]
    SUM --> FS[faithfulness_score]
```

对应函数：`extract_claims` → `verify_claim` → `faithfulness_score`

In [3]:
def extract_claims(answer: str, llm_client: OpenAI) -> list:
    """调用 DeepSeek 将答案拆解为原子性声明列表"""
    prompt = CLAIM_EXTRACTION_PROMPT.format(answer=answer)
    response = llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        claims = json.loads(raw)
        if isinstance(claims, list):
            return claims
    except json.JSONDecodeError:
        start = raw.find("[")
        end = raw.rfind("]") + 1
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end])
            except json.JSONDecodeError:
                pass
    return [raw]


def verify_claim(claim: str, context: str, llm_client: OpenAI) -> dict:
    """调用 DeepSeek 验证单条声明是否被上下文支持"""
    prompt = CLAIM_VERIFICATION_PROMPT.format(context=context, claim=claim)
    response = llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}") + 1
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end])
            except json.JSONDecodeError:
                pass
    supported = "true" in raw.lower()
    return {"supported": supported, "reason": raw}


def faithfulness_score(answer: str, context: str, llm_client: OpenAI) -> dict:
    """计算忠实度分数：拆解 → 逐条验证 → 汇总"""
    claims = extract_claims(answer, llm_client)
    verdicts = []
    for claim in claims:
        verdict = verify_claim(claim, context, llm_client)
        verdicts.append({"claim": claim, **verdict})

    supported_count = sum(1 for v in verdicts if v.get("supported", False))
    score = supported_count / len(claims) if claims else 0.0
    return {
        "score": score,
        "claims": verdicts,
        "total": len(claims),
        "supported": supported_count,
    }

## A/B/C 三场景预期

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    A[场景A 忠实] --> HA[Faithfulness 高]
    B[场景B 幻觉] --> HB[Faithfulness 低]
    C[场景C 部分忠实] --> HC[Faithfulness 中]
```

---
# Part 4 · 运行评估

对 A/B/C 三个场景依次执行 Faithfulness 评估，输出声明拆解、逐条验证结果与最终分数。

In [4]:
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

print("=" * 65)
print("Faithfulness 忠实度评估 - LLM-as-Judge 方法")
print("=" * 65)

for scenario in SCENARIOS:
    print(f"\n{'=' * 65}")
    print(f"  {scenario['name']}")
    print(f"{'=' * 65}")
    print(f"\n📄 上下文（节选）：{scenario['context'][:60]}...")
    print(f"\n💬 答案：{scenario['answer'][:60]}...")

    result = faithfulness_score(scenario["answer"], scenario["context"], client)

    print(f"\n🔍 原子性声明拆解与验证：")
    for i, v in enumerate(result["claims"], 1):
        icon = "✅" if v.get("supported") else "❌"
        print(f"  {i}. {icon} 声明：{v['claim']}")
        print(f"      原因：{v.get('reason', 'N/A')}")

    score = result["score"]
    bar = "█" * int(score * 10) + "░" * (10 - int(score * 10))
    print(f"\n📊 忠实度分数：{score:.2f}  [{bar}]")
    print(f"   支持 {result['supported']} / {result['total']} 条声明")

    if score >= 0.8:
        print("   评级：✅ 高忠实度 - 答案与上下文高度一致")
    elif score >= 0.5:
        print("   评级：⚠️  中等忠实度 - 存在部分未支持内容")
    else:
        print("   评级：❌ 低忠实度 - 答案包含大量幻觉内容")

print(f"\n{'=' * 65}")
print("💡 结论：幻觉场景评分低的原因：")
print("   答案中的事实声明（如发明年份、量子计算、使用率）")
print("   在上下文中均无依据，因此全部被判定为不支持。")
print("=" * 65)

Faithfulness 忠实度评估 - LLM-as-Judge 方法

  场景 A：忠实答案（高分预期）

📄 上下文（节选）：RAG 技术通过在生成前先检索相关文档，将外部知识注入到大语言模型的上下文中，从而减少幻觉、提升回答的准确性。向量数据库...

💬 答案：RAG 技术先检索相关文档，再注入到模型上下文中，通过引入外部知识来减少大语言模型的幻觉现象，提升准确性。...

🔍 原子性声明拆解与验证：
  1. ✅ 声明：RAG 技术先检索相关文档。
      原因：上下文明确指出RAG技术通过在生成前先检索相关文档来注入外部知识。
  2. ✅ 声明：RAG 技术将检索到的文档注入到模型上下文中。
      原因：上下文明确指出RAG技术通过在生成前先检索相关文档，将外部知识注入到大语言模型的上下文中。
  3. ✅ 声明：RAG 技术通过引入外部知识来减少大语言模型的幻觉现象。
      原因：上下文明确指出RAG技术通过检索相关文档将外部知识注入上下文，从而减少幻觉。
  4. ✅ 声明：RAG 技术通过引入外部知识来提升准确性。
      原因：上下文明确指出RAG技术通过检索相关文档将外部知识注入上下文，从而提升回答的准确性。

📊 忠实度分数：1.00  [██████████]
   支持 4 / 4 条声明
   评级：✅ 高忠实度 - 答案与上下文高度一致

  场景 B：幻觉答案（低分预期）

📄 上下文（节选）：RAG 技术通过在生成前先检索相关文档，将外部知识注入到大语言模型的上下文中，从而减少幻觉、提升回答的准确性。向量数据库...

💬 答案：RAG 技术于 2015 年由 OpenAI 发明，并在 GPT-4 中首次应用。它通过量子计算加速检索，每秒可处理 1...

🔍 原子性声明拆解与验证：
  1. ❌ 声明：RAG 技术于 2015 年由 OpenAI 发明。
      原因：上下文未提及RAG技术的发明时间或发明者，因此无法支持该声明。
  2. ❌ 声明：RAG 技术在 GPT-4 中首次应用。
      原因：上下文未提及RAG技术在GPT-4中首次应用，且RAG技术在此之前已被提出。
  3. ❌ 声明：RAG 技术通过量子计算加速检索。
      原因：上下文提到RAG技术使用向量数据库进行相